In [1]:
import sys
import os

print("Python:", sys.version)
print("CWD:", os.getcwd())
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")


Python: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
CWD: /home/void/Projects/EmbdAlys


'1'

In [2]:
import torch
import transformers
import safetensors
import huggingface_hub

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("safetensors:", safetensors.__version__)
print("huggingface_hub:", huggingface_hub.__version__)


torch: 2.10.0+cu129
transformers: 5.1.0
safetensors: 0.7.0
huggingface_hub: 1.4.1


In [4]:
!pip install sentencepiece
!pip install tiktoken

# tokenizer download
The AutoTokenizer is easy to fail. The most confidential way to download the model tokenizer is to manually download the file from the HuggingFace repo, and save it to local path.

- Mistral-7B: https://huggingface.co/mistralai/Mistral-7B-v0.1/tree/main
- Mixtral-8x7B: https://huggingface.co/mistralai/Mixtral-8x7B-v0.1/tree/main
- gpt-oss-20B: https://huggingface.co/openai/gpt-oss-20b/tree/main

The files we need:
- tokenizer_config.json
- tokenizer.json

To save the downloaded file in f"{model_name}/tokenizer", you can call the tokenizer by:
```
tokenizer_path = f"{model_name}/tokenizer" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
```

# embedding & unembedding download

In [3]:
from huggingface_hub import hf_hub_download
import json

model_name = "mistralai/Mistral-7B-v0.1"
fname = hf_hub_download(repo_id=model_name, filename="model.safetensors.index.json")

idx = json.load(open(fname, 'r'))

for tensor, loc in idx['weight_map'].items():
    if "lm_head.weight" in tensor or "embed_tokens.weight" in tensor:
        print(tensor, "→", loc)

lm_head.weight → model-00002-of-00002.safetensors
model.embed_tokens.weight → model-00001-of-00002.safetensors


In [8]:
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors import safe_open
import torch
import numpy as np
import os

# ===== Configuration =====
repo_id = "mistralai/Mistral-7B-v0.1"
shard_file_in = "model-00001-of-00002.safetensors"  
output_in = "mistralai/Mistral-7B-v0.1/tensors/input_embd.pt"

shard_file_out = "model-00002-of-00002.safetensors"  
output_out = "mistralai/Mistral-7B-v0.1/tensors/output_proj.pt"

# # ===== Step 1: Download tokenizer =====
# print("[*] Downloading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(repo_id)
# tokenizer.save_pretrained(tokenizer_dir)
# print(f"[✓] Tokenizer saved to {tokenizer_dir}")

# ===== Step 2: Download shard file for input embedding =====
print("[*] Downloading shard with input embedding...")
shard_path_in = hf_hub_download(repo_id=repo_id, filename=shard_file_in)
print(f"[✓] Shard downloaded: {shard_path_in}")

# ===== Step 3: Extract model.embed_tokens.weight =====
print("[*] Extracting model.embed_tokens.weight...")
with safe_open(shard_path_in, framework="pt", device="cpu") as f:
    input_embd = f.get_tensor("model.embed_tokens.weight")
print(f"[✓] Extracted shape: {input_embd.shape}")

# ===== Step 4: Save input_embd as .pt =====
torch.save(input_embd, output_in)
print(f"[✓] Saved to {output_in}")

# ===== Step 5: Process output projection =====
if shard_file_in == shard_file_out:
    print("[*] shard_file_in == shard_file_out, extracting output projection from same shard...")
    with safe_open(shard_path_in, framework="pt", device="cpu") as f:
        output_proj = f.get_tensor("lm_head.weight")
    print(f"[✓] Extracted shape: {output_proj.shape}")

    torch.save(output_proj, output_out)
    print(f"[✓] Saved to {output_out}")
else:
    if os.path.exists(shard_path_in):
        os.remove(shard_path_in)
        print(f"[✓] Deleted cached input shard: {shard_path_in}")

    print("[*] Downloading shard with output projection...")
    shard_path_out = hf_hub_download(repo_id=repo_id, filename=shard_file_out)
    print(f"[✓] Shard downloaded: {shard_path_out}")

    print("[*] Extracting lm_head.weight...")
    with safe_open(shard_path_out, framework="pt", device="cpu") as f:
        output_proj = f.get_tensor("lm_head.weight")
    print(f"[✓] Extracted shape: {output_proj.shape}")

    torch.save(output_proj, output_out)
    print(f"[✓] Saved to {output_out}")

    if os.path.exists(shard_path_out):
        os.remove(shard_path_out)
        print(f"[✓] Deleted cached output shard: {shard_path_out}")

[*] Downloading shard with input embedding...
[✓] Shard downloaded: /home/void/.cache/huggingface/hub/models--mistralai--Mistral-7B-v0.1/snapshots/27d67f1b5f57dc0953326b2601d68371d40ea8da/model-00001-of-00002.safetensors
[*] Extracting model.embed_tokens.weight...
[✓] Extracted shape: torch.Size([32000, 4096])
[✓] Saved to mistralai/Mistral-7B-v0.1/tensors/input_embd.pt
[✓] Deleted cached input shard: /home/void/.cache/huggingface/hub/models--mistralai--Mistral-7B-v0.1/snapshots/27d67f1b5f57dc0953326b2601d68371d40ea8da/model-00001-of-00002.safetensors
[*] Downloading shard with output projection...
[✓] Shard downloaded: /home/void/.cache/huggingface/hub/models--mistralai--Mistral-7B-v0.1/snapshots/27d67f1b5f57dc0953326b2601d68371d40ea8da/model-00002-of-00002.safetensors
[*] Extracting lm_head.weight...
[✓] Extracted shape: torch.Size([32000, 4096])
[✓] Saved to mistralai/Mistral-7B-v0.1/tensors/output_proj.pt
[✓] Deleted cached output shard: /home/void/.cache/huggingface/hub/models--mi

In [9]:
model_name = "mistralai/Mixtral-8x7B-v0.1"
fname = hf_hub_download(repo_id=model_name, filename="model.safetensors.index.json")

idx = json.load(open(fname, 'r'))

for tensor, loc in idx['weight_map'].items():
    if "lm_head.weight" in tensor or "embed_tokens.weight" in tensor:
        print(tensor, "→", loc)

lm_head.weight → model-00019-of-00019.safetensors
model.embed_tokens.weight → model-00001-of-00019.safetensors


In [12]:
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors import safe_open
import torch
import numpy as np
import os

# ===== Configuration =====
repo_id = "mistralai/Mixtral-8x7B-v0.1"
shard_file_in = "model-00001-of-00019.safetensors"  
output_in = "mistralai/Mixtral-8x7B-v0.1/tensors/input_embd.pt"

shard_file_out = "model-00019-of-00019.safetensors"  
output_out = "mistralai/Mixtral-8x7B-v0.1/tensors/output_proj.pt"

# # ===== Step 1: Download tokenizer =====
# print("[*] Downloading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(repo_id)
# tokenizer.save_pretrained(tokenizer_dir)
# print(f"[✓] Tokenizer saved to {tokenizer_dir}")

# ===== Step 2: Download shard file for input embedding =====
print("[*] Downloading shard with input embedding...")
shard_path_in = hf_hub_download(repo_id=repo_id, filename=shard_file_in)
print(f"[✓] Shard downloaded: {shard_path_in}")

# ===== Step 3: Extract model.embed_tokens.weight =====
print("[*] Extracting model.embed_tokens.weight...")
with safe_open(shard_path_in, framework="pt", device="cpu") as f:
    input_embd = f.get_tensor("model.embed_tokens.weight")
print(f"[✓] Extracted shape: {input_embd.shape}")

# ===== Step 4: Save input_embd as .pt =====
torch.save(input_embd, output_in)
print(f"[✓] Saved to {output_in}")

# ===== Step 5: Process output projection =====
if shard_file_in == shard_file_out:
    print("[*] shard_file_in == shard_file_out, extracting output projection from same shard...")
    with safe_open(shard_path_in, framework="pt", device="cpu") as f:
        output_proj = f.get_tensor("lm_head.weight")
    print(f"[✓] Extracted shape: {output_proj.shape}")

    torch.save(output_proj, output_out)
    print(f"[✓] Saved to {output_out}")
else:
    if os.path.exists(shard_path_in):
        os.remove(shard_path_in)
        print(f"[✓] Deleted cached input shard: {shard_path_in}")

    print("[*] Downloading shard with output projection...")
    shard_path_out = hf_hub_download(repo_id=repo_id, filename=shard_file_out)
    print(f"[✓] Shard downloaded: {shard_path_out}")

    print("[*] Extracting model.lm_head.weight...")
    with safe_open(shard_path_out, framework="pt", device="cpu") as f:
        output_proj = f.get_tensor("lm_head.weight")
    print(f"[✓] Extracted shape: {output_proj.shape}")

    torch.save(output_proj, output_out)
    print(f"[✓] Saved to {output_out}")

    if os.path.exists(shard_path_out):
        os.remove(shard_path_out)
        print(f"[✓] Deleted cached output shard: {shard_path_out}")

[*] Downloading shard with input embedding...
[✓] Shard downloaded: /home/void/.cache/huggingface/hub/models--mistralai--Mixtral-8x7B-v0.1/snapshots/fc7ac94680e38d7348cfa806e51218e6273104b0/model-00001-of-00019.safetensors
[*] Extracting model.embed_tokens.weight...
[✓] Extracted shape: torch.Size([32000, 4096])
[✓] Saved to mistralai/Mixtral-8x7B-v0.1/tensors/input_embd.pt
[✓] Deleted cached input shard: /home/void/.cache/huggingface/hub/models--mistralai--Mixtral-8x7B-v0.1/snapshots/fc7ac94680e38d7348cfa806e51218e6273104b0/model-00001-of-00019.safetensors
[*] Downloading shard with output projection...
[✓] Shard downloaded: /home/void/.cache/huggingface/hub/models--mistralai--Mixtral-8x7B-v0.1/snapshots/fc7ac94680e38d7348cfa806e51218e6273104b0/model-00019-of-00019.safetensors
[*] Extracting model.lm_head.weight...
[✓] Extracted shape: torch.Size([32000, 4096])
[✓] Saved to mistralai/Mixtral-8x7B-v0.1/tensors/output_proj.pt
[✓] Deleted cached output shard: /home/void/.cache/huggingfac

In [20]:
model_name = "openai/gpt-oss-20b"
fname = hf_hub_download(repo_id=model_name, filename="model.safetensors.index.json")

idx = json.load(open(fname, 'r'))

for tensor, loc in idx['weight_map'].items():
    if "lm_head.weight" in tensor or "embed_tokens.weight" in tensor:
        print(tensor, "→", loc)

model.embed_tokens.weight → model-00002-of-00002.safetensors
lm_head.weight → model-00002-of-00002.safetensors


In [14]:
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors import safe_open
import torch
import numpy as np
import os

# ===== Configuration =====
repo_id = "openai/gpt-oss-20b"
shard_file_in = "model-00002-of-00002.safetensors"  
output_in = "gpt-oss/tensors/input_embd.pt"

shard_file_out = "model-00002-of-00002.safetensors"  
output_out = "gpt-oss/tensors/output_proj.pt"

# # ===== Step 1: Download tokenizer =====
# print("[*] Downloading tokenizer...")
# tokenizer = AutoTokenizer.from_pretrained(repo_id)
# tokenizer.save_pretrained(tokenizer_dir)
# print(f"[✓] Tokenizer saved to {tokenizer_dir}")

# ===== Step 2: Download shard file for input embedding =====
print("[*] Downloading shard with input embedding...")
shard_path_in = hf_hub_download(repo_id=repo_id, filename=shard_file_in)
print(f"[✓] Shard downloaded: {shard_path_in}")

# ===== Step 3: Extract model.embed_tokens.weight =====
print("[*] Extracting model.embed_tokens.weight...")
with safe_open(shard_path_in, framework="pt", device="cpu") as f:
    input_embd = f.get_tensor("model.embed_tokens.weight")
print(f"[✓] Extracted shape: {input_embd.shape}")

# ===== Step 4: Save input_embd as .pt =====
torch.save(input_embd, output_in)
print(f"[✓] Saved to {output_in}")

# ===== Step 5: Process output projection =====
if shard_file_in == shard_file_out:
    print("[*] shard_file_in == shard_file_out, extracting output projection from same shard...")
    with safe_open(shard_path_in, framework="pt", device="cpu") as f:
        output_proj = f.get_tensor("lm_head.weight")
    print(f"[✓] Extracted shape: {output_proj.shape}")

    torch.save(output_proj, output_out)
    print(f"[✓] Saved to {output_out}")
else:
    if os.path.exists(shard_path_in):
        os.remove(shard_path_in)
        print(f"[✓] Deleted cached input shard: {shard_path_in}")

    print("[*] Downloading shard with output projection...")
    shard_path_out = hf_hub_download(repo_id=repo_id, filename=shard_file_out)
    print(f"[✓] Shard downloaded: {shard_path_out}")

    print("[*] Extracting lm_head.weight...")
    with safe_open(shard_path_out, framework="pt", device="cpu") as f:
        output_proj = f.get_tensor("lm_head.weight")
    print(f"[✓] Extracted shape: {output_proj.shape}")

    torch.save(output_proj, output_out)
    print(f"[✓] Saved to {output_out}")

    if os.path.exists(shard_path_out):
        os.remove(shard_path_out)
        print(f"[✓] Deleted cached output shard: {shard_path_out}")

[*] Downloading shard with input embedding...
[✓] Shard downloaded: /home/void/.cache/huggingface/hub/models--openai--gpt-oss-20b/snapshots/6cee5e81ee83917806bbde320786a8fb61efebee/model-00002-of-00002.safetensors
[*] Extracting model.embed_tokens.weight...
[✓] Extracted shape: torch.Size([201088, 2880])
[✓] Saved to gpt-oss/tensors/input_embd.pt
[*] shard_file_in == shard_file_out, extracting output projection from same shard...
[✓] Extracted shape: torch.Size([201088, 2880])
[✓] Saved to gpt-oss/tensors/output_proj.pt


# -----------Archived-----------

In [11]:
from dump_hf_tensors import main


In [12]:
args = [
    "--model_name", MODEL_NAME,
    "--out_dir", OUT_DIR,
    "--cache_dir", CACHE_DIR,
    # Default：save_tokenizer=True, save_config=True, spaces=all
]

ret = main(args)
print("Return code:", ret)


ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [14]:
import os
from pathlib import Path

root = Path(OUT_DIR) / MODEL_NAME

expected_paths = [
    root / "tokenizer",
    root / "tensors" / "input_embd.pt",
    root / "tensors" / "output_proj.pt",
    root / "config" / "config.json",
    root / "meta" / "resolved.json",
]

for p in expected_paths:
    print(p, "->", "OK" if p.exists() else "MISSING")


artifacts_test/mistralai/Mixtral-8x7B-v0.1/tokenizer -> OK
artifacts_test/mistralai/Mixtral-8x7B-v0.1/tensors/input_embd.pt -> OK
artifacts_test/mistralai/Mixtral-8x7B-v0.1/tensors/output_proj.pt -> OK
artifacts_test/mistralai/Mixtral-8x7B-v0.1/config/config.json -> OK
artifacts_test/mistralai/Mixtral-8x7B-v0.1/meta/resolved.json -> OK


In [1]:
import torch

embd = torch.load(root / "tensors" / "input_embd.pt", map_location="cpu")
proj = torch.load(root / "tensors" / "output_proj.pt", map_location="cpu")

print("input_embd shape:", embd.shape, embd.dtype)
print("output_proj shape:", proj.shape, proj.dtype)

NameError: name 'root' is not defined

In [16]:
import json

with open(root / "meta" / "resolved.json", "r") as f:
    meta = json.load(f)

meta.keys(), meta["outputs"]


(dict_keys(['timestamp_utc', 'model_name', 'revision', 'family', 'spaces_arg', 'save_tokenizer', 'save_config', 'extra_tensors_arg', 'root', 'outputs', 'resolved']),
 [{'type': 'tokenizer',
   'path': './artifacts_test/mistralai/Mixtral-8x7B-v0.1/tokenizer'},
  {'type': 'config',
   'path': './artifacts_test/mistralai/Mixtral-8x7B-v0.1/config/config.json'},
  {'type': 'tensor',
   'logical_name': 'input_embd',
   'path': './artifacts_test/mistralai/Mixtral-8x7B-v0.1/tensors/input_embd.pt'},
  {'type': 'tensor',
   'logical_name': 'output_proj',
   'path': './artifacts_test/mistralai/Mixtral-8x7B-v0.1/tensors/output_proj.pt'}])

In [17]:
for rec in meta["resolved"]:
    print(
        rec.get("logical_name"),
        "->",
        rec.get("resolved_tensor_key"),
        "| shape =", rec.get("shape"),
    )


input_embd -> model.embed_tokens.weight | shape = [32000, 4096]
output_proj -> lm_head.weight | shape = [32000, 4096]


In [18]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(root / "tokenizer")

print("Tokenizer vocab size:", tok.vocab_size)
print("Special tokens:", tok.special_tokens_map)


Tokenizer vocab size: 32000
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}


In [4]:
from dump_hf_tensors import main

args = [
    "--model_name", "EleutherAI/pythia-1b",
    "--family", "pythia",
    "--out_dir", "./artifacts_test",
    "--cache_dir", "./.hf_cache",
    "--spaces", "all",
    "--steps", "all",
    # tokenizer/config 默认会保存一次到 base_root，不需要额外参数
]

rc = main(args)
print("Return code:", rc)

[OK] Export completed. Root: ./artifacts_test/EleutherAI/pythia-1b/step0
[OK] Meta: ./artifacts_test/EleutherAI/pythia-1b/step0/meta/resolved.json
[OK] downloads/ cleaned.
[OK] Export completed. Root: ./artifacts_test/EleutherAI/pythia-1b/step1
[OK] Meta: ./artifacts_test/EleutherAI/pythia-1b/step1/meta/resolved.json
[OK] downloads/ cleaned.
[OK] Export completed. Root: ./artifacts_test/EleutherAI/pythia-1b/step2
[OK] Meta: ./artifacts_test/EleutherAI/pythia-1b/step2/meta/resolved.json
[OK] downloads/ cleaned.
[OK] Export completed. Root: ./artifacts_test/EleutherAI/pythia-1b/step4
[OK] Meta: ./artifacts_test/EleutherAI/pythia-1b/step4/meta/resolved.json
[OK] downloads/ cleaned.
[OK] Export completed. Root: ./artifacts_test/EleutherAI/pythia-1b/step8
[OK] Meta: ./artifacts_test/EleutherAI/pythia-1b/step8/meta/resolved.json
[OK] downloads/ cleaned.
[OK] Export completed. Root: ./artifacts_test/EleutherAI/pythia-1b/step16
[OK] Meta: ./artifacts_test/EleutherAI/pythia-1b/step16/meta/resol

In [1]:
import sys
import platform
import subprocess

def safe_import(pkg_name):
    try:
        module = __import__(pkg_name)
        return module, getattr(module, "__version__", "unknown")
    except Exception as e:
        return None, f"NOT INSTALLED ({e})"


def get_torch_cuda_info():
    try:
        import torch

        cuda_available = torch.cuda.is_available()
        cuda_version = torch.version.cuda
        torch_version = torch.__version__

        if cuda_available:
            device_name = torch.cuda.get_device_name(0)
            total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
        else:
            device_name = None
            total_mem = None

        return {
            "torch_version": torch_version,
            "cuda_available": cuda_available,
            "cuda_version": cuda_version,
            "gpu_name": device_name,
            "gpu_mem_gb": total_mem,
        }
    except Exception as e:
        return {"error": str(e)}


def get_nvidia_smi():
    try:
        result = subprocess.check_output(["nvidia-smi"], stderr=subprocess.STDOUT)
        return result.decode("utf-8")
    except Exception as e:
        return f"nvidia-smi not available ({e})"


def check_cuml_hdbscan():
    try:
        import cuml
        from cuml.cluster import HDBSCAN as cuHDBSCAN
        return f"cuML HDBSCAN available (RAPIDS {cuml.__version__})"
    except Exception:
        return "cuML HDBSCAN NOT available"

def check_cpu_hdbscan():
    try:
        import hdbscan
        return f"CPU HDBSCAN available (version {hdbscan.__version__})"
    except Exception:
        return "CPU HDBSCAN NOT available"


def main():
    print("=" * 60)
    print("SYSTEM")
    print("=" * 60)
    print(f"OS: {platform.system()} {platform.release()}")
    print(f"Python: {sys.version}")

    print("\n" + "=" * 60)
    print("CORE LIBRARIES")
    print("=" * 60)

    libs = [
        "numpy",
        "pandas",
        "sklearn",
        "torch",
        "transformers",
        "matplotlib",
    ]

    for lib in libs:
        _, ver = safe_import(lib)
        print(f"{lib}: {ver}")

    print("\n" + "=" * 60)
    print("HDBSCAN")
    print("=" * 60)
    print(check_cpu_hdbscan())
    print(check_cuml_hdbscan())

    print("\n" + "=" * 60)
    print("CUDA / GPU (PyTorch)")
    print("=" * 60)
    torch_info = get_torch_cuda_info()
    for k, v in torch_info.items():
        print(f"{k}: {v}")

    print("\n" + "=" * 60)
    print("NVIDIA-SMI")
    print("=" * 60)
    print(get_nvidia_smi())

In [2]:
main()

SYSTEM
OS: Linux 6.8.0-64-generic
Python: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]

CORE LIBRARIES
numpy: 2.2.6
pandas: 2.3.3
sklearn: 1.7.2
torch: 2.10.0+cu129
transformers: 5.1.0
matplotlib: 3.10.8

HDBSCAN
CPU HDBSCAN NOT available
cuML HDBSCAN available (RAPIDS 25.12.00)

CUDA / GPU (PyTorch)
torch_version: 2.10.0+cu129
cuda_available: True
cuda_version: 12.9
gpu_name: NVIDIA GeForce RTX 4070 Ti SUPER
gpu_mem_gb: 15.53680419921875

NVIDIA-SMI
Thu Mar 19 00:23:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |    